# NPTEL Study Notes: Knowledge Graph – Modeling Hierarchies

These notes cover the key concepts, mathematical formulations, and embedding techniques used to model type hierarchies in knowledge graphs, as discussed in the "Introduction to Large Language Models (LLMs)" lecture.

---

## 1. Introduction to Type Hierarchies in Knowledge Graphs

While earlier models (like TransE, RotatE) focus on general relations in triples $(h, r, t)$, another critical property of relations is **transitivity**.

* **Transitive Relation:** If $A \rightarrow B$ and $B \rightarrow C$, then $A \rightarrow C$ (e.g., Camel is a Mammal, Mammal is an Animal $\Rightarrow$ Camel is an Animal).
* **Application:** Understanding transitivity and hierarchical type containment is essential for Question Answering and interfacing with Large Language Models (LLMs), as answers often rely on implicit ontological knowledge.
* **Structure:** Type hierarchies are generally **Directed Acyclic Graphs (DAGs)**, not strict trees. Nodes can have multiple parents and fan out to multiple children. The root represents all entities, branching into finer, specialized types.
* **Popular Type Systems:** Figer (Google, ~100 types), DBpedia (~700 types), WordNet (~15,000 types), and Typenet (~1,000 types).
* **Limitation of Standard Models:** Techniques like translation, rotation, and factorization are generally inadequate for explicitly capturing these hierarchical graph structures.

---

## 2. Challenges with Euclidean Embeddings

Embedding a hierarchical DAG into a vector space aims to preserve graph distances as spatial distances. However, Euclidean space struggles with trees.

* **Exponential Expansion:** In a tree, the number of leaf nodes grows exponentially. Maintaining accurate distances between a parent and its numerous children requires vast spatial capacity near the periphery, which Euclidean space cannot easily accommodate without excessive dimensionality.

### 2.1 Low-Distortion Embeddings

Distortion measures how poorly an embedding space preserves the original graph distances.

* **Distortion Formula:**

$$\text{Distortion} = \frac{\max_{u,v \in G} \left( \frac{d_{\text{embed}}(X_u, X_v)}{d_{G}(u, v)} \right)}{\min_{u,v \in G} \left( \frac{d_{\text{embed}}(X_u, X_v)}{d_{G}(u, v)} \right)}$$



Where $d_{\text{embed}}$ is the distance in the embedding space and $d_G$ is the graph distance (e.g., shortest path).
* **Theoretical Hardness in Euclidean Space:**
* Arbitrary graph ($n$ nodes, large $d$): $O(\log n)$ distortion [1].
* Planar graph in 2D ($R^2$): $O(n)$ distortion.
* Arbitrary tree: $O(\sqrt{n})$ distortion.
* Tree in large dimensionality $R^d$: $O(\log \log n)$ distortion [1].
* Binary tree in 1D line: $O(n / \log n)$ distortion.
* *Note:* A constant multiplicative error of $(1 + \epsilon)$ is **impossible** in Euclidean space, but possible in hyperbolic space.



---

## 3. Hyperbolic Embeddings (Poincaré Ball)

Hyperbolic space inherently expands exponentially, mirroring the structure of trees.

* **Hyperbolic Distance:** All points are embedded strictly inside a unit circle (disk). The distance is measured along a geodesic (hyperbola) rather than a straight line.

$$d_H(x, y) = \text{arccosh}\left( 1 + 2 \frac{\vert{}\vert{}x - y\vert{}\vert{}^2}{(1 - \vert{}\vert{}x\vert{}\vert{}^2)(1 - \vert{}\vert{}y\vert{}\vert{}^2)} \right)$$


* **Hyperbolic Arccosine:** $\text{arccosh}(a) = \ln(a + \sqrt{a^2 - 1})$
* **Key Property:** As vectors $x$ or $y$ approach the perimeter of the disk, $\vert{}\vert{}x\vert{}\vert{} \rightarrow 1$ and $\vert{}\vert{}y\vert{}\vert{} \rightarrow 1$. The denominator approaches $0$, causing $d_H(x, y) \rightarrow \infty$. While the disk has a finite perimeter in Euclidean space, it has an **infinite perimeter** in hyperbolic space, providing ample "room" to embed exponentially growing leaf nodes.
* **Limitations:**
* Computers use finite precision bits, making it hard to properly represent points infinitely close to the boundary.
* Hyperbolic embeddings do not seamlessly integrate with standard Euclidean word/entity embeddings used in language models.



---

## 4. Interval Containment and Vector Dominance

Instead of relying on distance metrics (Euclidean or Hyperbolic), hierarchies can be modeled by checking for topological containment.

### 4.1 In-Order Traversal (1D Embedding)

* By performing an in-order traversal of a tree, each node $X$ is assigned an entry time and an exit time, forming an interval $I(X) = [\text{enter}, \text{exit}]$.
* **Containment Test:** Node $X$ is a descendant of node $Y$ if and only if $I(X) \subset I(Y)$. This allows checking hierarchical relations in $O(1)$ constant time.

### 4.2 Cone Embeddings (Order Embeddings)

This generalizes the 1D interval concept to multi-dimensional **vector dominance**.

* **Vector Dominance:** A type is represented by an apex vector $u$. $X$ is a subtype (descendant) of $Y$ ($X \preceq Y$) if $u_X \ge u_Y$ element-wise.
* **Geometric Interpretation:** The type forms an upward-open cone. The supertype's cone completely envelops the subtype's cone.
* **Training Loss Function:**
* **Positive Loss ($L_+$):** When $X$ *should* be a subtype of $Y$. We want $u_{X,d} \ge u_{Y,d}$. If $u_{Y,d} - u_{X,d} > 0$, we incur a penalty.

$$L_+ = \max_d \max(0, u_{Y,d} - u_{X,d})$$


* **Negative Loss ($L_-$):** When $X$ *should not* be a subtype of $Y$. There must be *at least one* dimension where $X$ violates containment (i.e., $u_{X,d} < u_{Y,d}$). A margin $\alpha$ is often used.


* **Limitations of Open Cones:**
* Because the cones open upward to infinity, their volume is infinite. A subtype (e.g., Politicians) ends up having the same infinite volume as its supertype (People), which is illogical since the cardinality of a subset is strictly smaller.
* Cannot effectively model **negative correlations** or disjoint sets (e.g., mutually exclusive types like "Fruit" and "Scientist").



### 4.3 Box Embeddings

To solve the infinite volume and disjoint set issues, open cones are generalized to **closed hyperrectangles (boxes)**.

* Each type $X$ is parameterized by an interval in every dimension $d$, defined by a base $B_{X,d}$ and a height $H_{X,d}$.
* Type-to-type containment translates to strict multi-dimensional interval containment (like overlapping Venn diagrams), allowing for finite volumes and clearly separated disjoint types.

---

## Practice Questions

**Q1: Why is standard Euclidean space poorly suited for embedding hierarchical tree structures like WordNet?**
**Answer:** In a hierarchical tree, the number of leaf nodes grows exponentially with depth. Euclidean space expands polynomially, meaning there isn't enough spatial capacity near the periphery to embed the nodes while maintaining their correct graph distances without severe distortion.

**Q2: In hyperbolic embeddings, what happens to the distance between two points as they move closer to the perimeter of the unit disk?**
**Answer:** The hyperbolic distance approaches infinity. The $(1 - \vert{}\vert{}x\vert{}\vert{}^2)$ and $(1 - \vert{}\vert{}y\vert{}\vert{}^2)$ terms in the denominator approach zero as the norms approach 1, driving the entire distance calculation toward infinity.

**Q3: Explain how vector dominance works in the context of Cone (Order) embeddings for the relation "Actor is a subtype of Person".**
**Answer:** Each type is represented by an apex vector in $d$ dimensions. For "Actor" to be a subtype of "Person", the vector for Actor ($u_{\text{actor}}$) must dominate the vector for Person ($u_{\text{person}}$) element-wise. This means for every dimension $d$, $u_{\text{actor}, d} \ge u_{\text{person}, d}$. Geometrically, the cone originating from the Actor apex is strictly contained inside the cone originating from the Person apex.

**Q4: What are the two main limitations of Cone Embeddings that Box Embeddings solve?**
**Answer:**

1. **Infinite Volume:** Cones are open upwards, giving subsets the same infinite volume as their super-sets. Box embeddings use bounded hyperrectangles to represent finite volumes.
2. **Disjoint Types:** Cones that expand upward inevitably overlap, making it impossible to model disjoint, mutually exclusive concepts. Box embeddings can be placed as non-overlapping rectangles.

---

## Revision Summary

* **Transitivity** in graphs creates Type Hierarchies (DAGs). Standard translation models fail to capture this topological data natively.
* **Distortion Limits:** Euclidean spaces impose $O(\log n)$ to $O(\log \log n)$ distortions.
* **Hyperbolic Space (Poincaré Ball):** Exponentially expands, giving infinite perimeter room for tree leaves, but suffers from finite precision on standard hardware.
* **Interval Traversal:** 1D Topological containment $I(X) \subset I(Y)$ answers descendant queries in $O(1)$ time without distance metrics.
* **Cone Embeddings:** Uses vector dominance $u_X \ge u_Y$. Fails at modeling disjoint types and subset volume scaling.
* **Box Embeddings:** Generalizes cones to closed hyperrectangles, resolving the volume and intersection limits of open cones.